# Ultimate Hybrid Reddit Dating Questions Extractor

## Clean, Working Implementation

This notebook provides a fully functional hybrid Reddit question extractor that:
- Extracts high-quality dating questions from Reddit
- Uses intelligent hybrid scoring for question quality
- Includes all required output columns
- Works with or without Reddit API credentials
- Handles errors gracefully

### Features:
- **Hybrid Scoring Model**: Combines similarity, framework, depth, engagement, and personal connection scores
- **Smart Filtering**: Only dating-relevant, high-quality questions
- **Multiple Output Formats**: Excel and CSV export
- **Robust Error Handling**: Works in demo mode if Reddit API unavailable
- **Comprehensive Metadata**: All required columns plus detailed scoring breakdown

## Step 1: Install Required Dependencies

Run this cell first to install all necessary packages.

In [8]:
# Install required packages
import subprocess
import sys

def install_package(package):
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])
        print(f"✅ Successfully installed {package}")
    except subprocess.CalledProcessError:
        print(f"❌ Failed to install {package}")

# Core dependencies
packages = [
    "pandas>=1.3.0",
    "numpy>=1.21.0", 
    "scikit-learn>=1.0.0",
    "openpyxl>=3.0.0",
    "praw>=7.0.0"  # Optional for Reddit API
]

print("🚀 Installing dependencies...")
for package in packages:
    install_package(package)

print("✅ Installation complete!")

🚀 Installing dependencies...
✅ Successfully installed pandas>=1.3.0
✅ Successfully installed numpy>=1.21.0
✅ Successfully installed scikit-learn>=1.0.0
✅ Successfully installed openpyxl>=3.0.0
✅ Successfully installed praw>=7.0.0
✅ Installation complete!


## Step 2: Import Libraries and Configuration

In [9]:
# Core libraries
import pandas as pd
import numpy as np
import re
import json
import time
import hashlib
from datetime import datetime, timedelta
from pathlib import Path
from typing import List, Dict, Any, Optional, Tuple
from collections import Counter
import random
import warnings
warnings.filterwarnings('ignore')

# Try to import optional dependencies
try:
    import praw
    PRAW_AVAILABLE = True
    print("✅ PRAW (Reddit API) available")
except ImportError:
    PRAW_AVAILABLE = False
    print("⚠️ PRAW not available - will use demo mode")

try:
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.metrics.pairwise import cosine_similarity
    SKLEARN_AVAILABLE = True
    print("✅ scikit-learn available")
except ImportError:
    SKLEARN_AVAILABLE = False
    print("⚠️ scikit-learn not available - using basic scoring")

print("📚 All libraries imported successfully!")

✅ PRAW (Reddit API) available
✅ scikit-learn available
📚 All libraries imported successfully!


## Step 3: Configuration Settings

Adjust these settings as needed for your extraction requirements.

In [10]:
# Configuration
CONFIG = {
    'min_hybrid_score': 65,           # Minimum quality score (0-100)
    'min_question_length': 15,        # Minimum characters
    'max_question_length': 250,       # Maximum characters
    'posts_per_subreddit': 50,        # Posts to check per subreddit
    'max_questions_total': 150,       # Maximum final questions
    'min_post_score': 20,             # Minimum Reddit post score
    'min_comment_score': 5,           # Minimum comment score
    'output_dir': './outputs',        # Output directory
    'verbose': True                   # Show detailed progress
}

# Subreddits to search for dating questions
SUBREDDITS = [
    'AskReddit', 'dating_advice', 'relationships', 'relationship_advice',
    'CasualConversation', 'socialskills', 'AskWomen', 'AskMen',
    'WouldYouRather', 'hypotheticalsituation', 'dating',
    'OkCupid', 'hingeapp', 'bumble', 'tinder'
]

# Dating-related keywords for filtering
DATING_KEYWORDS = [
    'date', 'dating', 'relationship', 'partner', 'boyfriend', 'girlfriend',
    'love', 'romance', 'attraction', 'crush', 'flirt', 'match', 'single',
    'couple', 'marriage', 'wedding', 'engagement', 'valentine', 'anniversary'
]

# Reddit API configuration (optional)
# Uncomment and fill in your credentials if you have them
REDDIT_CONFIG = None
# REDDIT_CONFIG = {
#     'client_id': 'your_client_id',
#     'client_secret': 'your_client_secret',
#     'user_agent': 'HybridExtractor/1.0'
# }

# Create output directory
Path(CONFIG['output_dir']).mkdir(parents=True, exist_ok=True)

print("⚙️ Configuration loaded successfully!")
print(f"🎯 Target score: ≥{CONFIG['min_hybrid_score']}")
print(f"🌐 Subreddits: {len(SUBREDDITS)}")
print(f"💾 Output: {CONFIG['output_dir']}")

⚙️ Configuration loaded successfully!
🎯 Target score: ≥65
🌐 Subreddits: 15
💾 Output: ./outputs


## Step 4: Hybrid Scoring Model

This class implements the intelligent scoring system for question quality assessment.

In [11]:
class HybridScoringModel:
    """Hybrid scoring model for question quality assessment"""
    
    def __init__(self, core_questions: List[str] = None):
        self.core_questions = core_questions or self._get_default_core_questions()
        
        if SKLEARN_AVAILABLE and self.core_questions:
            self.vectorizer = TfidfVectorizer(
                max_features=500,
                stop_words='english',
                ngram_range=(1, 2),
                lowercase=True
            )
            processed_questions = [self._preprocess_text(q) for q in self.core_questions]
            self.core_vectors = self.vectorizer.fit_transform(processed_questions)
        else:
            self.vectorizer = None
            self.core_vectors = None
        
        print(f"🤖 Hybrid model initialized with {len(self.core_questions)} core questions")
    
    def _get_default_core_questions(self) -> List[str]:
        """Default set of high-quality dating questions"""
        return [
            "What's your idea of a perfect first date?",
            "What are you most passionate about in life?",
            "What's something you've always wanted to try but haven't yet?",
            "What's your biggest goal for this year?",
            "What makes you laugh the most?",
            "What's your favorite way to spend a weekend?",
            "What's something you're really proud of?",
            "What's your love language?",
            "What's the best advice you've ever received?",
            "What's something that always makes you smile?",
            "What's your biggest pet peeve in dating?",
            "What's your ideal relationship like?",
            "What's something you can't live without?",
            "What's your favorite memory from childhood?",
            "What's something you're looking forward to?",
            "What's your biggest fear in relationships?",
            "What's something that instantly attracts you to someone?",
            "What's your definition of a successful relationship?",
            "What's something you wish more people knew about you?",
            "What's your favorite thing about yourself?"
        ]
    
    def _preprocess_text(self, text: str) -> str:
        """Basic text preprocessing"""
        text = text.lower()
        text = re.sub(r'[^\w\s]', '', text)
        return text.strip()
    
    def calculate_hybrid_score(self, question_text: str) -> Dict[str, Any]:
        """Calculate hybrid score for a question"""
        try:
            # Similarity to core questions (if available)
            similarity_score = 50  # Default
            if self.vectorizer and self.core_vectors is not None:
                processed_q = self._preprocess_text(question_text)
                q_vector = self.vectorizer.transform([processed_q])
                similarities = cosine_similarity(q_vector, self.core_vectors)[0]
                max_similarity = np.max(similarities)
                similarity_score = min(100, max_similarity * 100)
            
            # Pattern-based scoring
            framework_score = self._calculate_framework_score(question_text)
            depth_score = self._calculate_depth_score(question_text)
            engagement_score = self._calculate_engagement_score(question_text)
            personal_score = self._calculate_personal_score(question_text)
            
            # Weighted final score
            final_score = (
                similarity_score * 0.40 +
                framework_score * 0.20 +
                depth_score * 0.15 +
                engagement_score * 0.15 +
                personal_score * 0.10
            )
            
            return {
                'hybrid_score': round(final_score, 2),
                'similarity_to_core': round(similarity_score, 2),
                'framework_match': round(framework_score, 2),
                'depth_potential': round(depth_score, 2),
                'engagement_potential': round(engagement_score, 2),
                'personal_connection': round(personal_score, 2)
            }
        except Exception as e:
            print(f"Error calculating score: {e}")
            return {
                'hybrid_score': 50.0,
                'similarity_to_core': 50.0,
                'framework_match': 50.0,
                'depth_potential': 50.0,
                'engagement_potential': 50.0,
                'personal_connection': 50.0
            }
    
    def _calculate_framework_score(self, text: str) -> float:
        """Score based on question framework patterns"""
        score = 50
        text_lower = text.lower()
        
        # Question starters that work well
        good_starters = ['what', 'how', 'why', 'when', 'where', 'which', 'who']
        if any(text_lower.startswith(starter) for starter in good_starters):
            score += 20
        
        # Avoid yes/no questions
        if text_lower.startswith(('do you', 'are you', 'have you', 'can you', 'will you')):
            score -= 15
        
        # Prefer open-ended questions
        if any(word in text_lower for word in ['favorite', 'best', 'worst', 'most', 'least']):
            score += 15
        
        return min(100, max(0, score))
    
    def _calculate_depth_score(self, text: str) -> float:
        """Score based on potential for deep conversation"""
        score = 50
        text_lower = text.lower()
        
        # Words that indicate depth
        depth_words = ['feel', 'think', 'believe', 'value', 'important', 'meaningful', 
                      'experience', 'learn', 'grow', 'change', 'future', 'past', 'dream']
        depth_count = sum(1 for word in depth_words if word in text_lower)
        score += depth_count * 10
        
        # Avoid surface-level questions
        surface_words = ['color', 'food', 'movie', 'song', 'book']
        if any(word in text_lower for word in surface_words):
            score -= 10
        
        return min(100, max(0, score))
    
    def _calculate_engagement_score(self, text: str) -> float:
        """Score based on engagement potential"""
        score = 50
        text_lower = text.lower()
        
        # Engaging question types
        if any(phrase in text_lower for phrase in ['would you rather', 'if you could', 'imagine']):
            score += 20
        
        # Personal relevance
        if any(word in text_lower for word in ['you', 'your', 'yourself']):
            score += 15
        
        # Length consideration (not too short, not too long)
        length = len(text)
        if 20 <= length <= 100:
            score += 10
        elif length > 150:
            score -= 10
        
        return min(100, max(0, score))
    
    def _calculate_personal_score(self, text: str) -> float:
        """Score based on personal connection potential"""
        score = 50
        text_lower = text.lower()
        
        # Personal sharing indicators
        personal_words = ['personal', 'share', 'tell', 'story', 'experience', 
                         'memory', 'feel', 'emotion', 'secret', 'private']
        personal_count = sum(1 for word in personal_words if word in text_lower)
        score += personal_count * 8
        
        # Dating-specific personal topics
        dating_personal = ['relationship', 'love', 'attraction', 'partner', 'date']
        if any(word in text_lower for word in dating_personal):
            score += 15
        
        return min(100, max(0, score))

print("✅ HybridScoringModel class defined successfully!")

✅ HybridScoringModel class defined successfully!


## Step 5: Reddit Extractor Class

This class handles Reddit API interaction and question extraction.

In [12]:
class RedditExtractor:
    """Reddit question extractor with hybrid scoring"""
    
    def __init__(self, reddit_config: Dict = None):
        self.reddit = None
        self.scoring_model = HybridScoringModel()
        
        if PRAW_AVAILABLE and reddit_config:
            try:
                self.reddit = praw.Reddit(**reddit_config)
                print("✅ Reddit API connected successfully")
            except Exception as e:
                print(f"❌ Reddit API connection failed: {e}")
                self.reddit = None
    
    def extract_questions(self) -> pd.DataFrame:
        """Main extraction method"""
        if self.reddit:
            return self._extract_from_reddit()
        else:
            print("🔄 Using demo mode - generating sample questions")
            return self._generate_demo_questions()
    
    def _extract_from_reddit(self) -> pd.DataFrame:
        """Extract questions from Reddit using PRAW"""
        all_questions = []
        
        for subreddit_name in SUBREDDITS:
            if CONFIG['verbose']:
                print(f"🔍 Searching r/{subreddit_name}...")
            
            try:
                subreddit = self.reddit.subreddit(subreddit_name)
                posts = list(subreddit.hot(limit=CONFIG['posts_per_subreddit']))
                
                for post in posts:
                    # Skip if post score too low
                    if post.score < CONFIG['min_post_score']:
                        continue
                    
                    # Check post title
                    if self._is_question(post.title):
                        question_data = self._process_question(
                            post.title, subreddit_name, post.score, 
                            post.num_comments, post.url, 'post'
                        )
                        if question_data:
                            all_questions.append(question_data)
                    
                    # Check post text
                    if hasattr(post, 'selftext') and post.selftext:
                        questions_in_text = self._extract_questions_from_text(post.selftext)
                        for q in questions_in_text:
                            question_data = self._process_question(
                                q, subreddit_name, post.score, 
                                post.num_comments, post.url, 'post_text'
                            )
                            if question_data:
                                all_questions.append(question_data)
                    
                    # Check comments
                    try:
                        post.comments.replace_more(limit=0)
                        for comment in post.comments[:10]:  # Top 10 comments
                            if comment.score >= CONFIG['min_comment_score']:
                                if self._is_question(comment.body):
                                    question_data = self._process_question(
                                        comment.body, subreddit_name, comment.score,
                                        0, post.url, 'comment'
                                    )
                                    if question_data:
                                        all_questions.append(question_data)
                    except Exception as e:
                        if CONFIG['verbose']:
                            print(f"  ⚠️ Error processing comments: {e}")
                        continue
                
                if CONFIG['verbose']:
                    count = len([q for q in all_questions if q['reddit_topic'] == subreddit_name])
                    print(f"  ✅ Found {count} questions")
                
            except Exception as e:
                print(f"❌ Error processing r/{subreddit_name}: {e}")
                continue
        
        return self._create_dataframe(all_questions)
    
    def _generate_demo_questions(self) -> pd.DataFrame:
        """Generate demo questions when Reddit API is not available"""
        demo_questions = [
            "What's the most important quality you look for in a partner?",
            "How do you know when you're ready for a serious relationship?",
            "What's your biggest dating red flag?",
            "What's the best date you've ever been on?",
            "How do you handle disagreements in relationships?",
            "What's something you wish you knew before you started dating?",
            "What's your love language and how did you discover it?",
            "What's the most romantic gesture someone has done for you?",
            "How do you maintain your independence in a relationship?",
            "What's your ideal way to spend a weekend with a partner?",
            "What's something you're not willing to compromise on in dating?",
            "How do you know if someone is genuinely interested in you?",
            "What's the best relationship advice you've ever received?",
            "What's your biggest fear when it comes to dating?",
            "How do you balance career and relationship goals?",
            "What's something that instantly makes you lose interest in someone?",
            "What's your definition of emotional intimacy?",
            "How do you handle long-distance relationships?",
            "What's the most important lesson you've learned from past relationships?",
            "What's your approach to introducing someone to your family and friends?"
        ]
        
        all_questions = []
        for i, question in enumerate(demo_questions):
            question_data = self._process_question(
                question, 'demo', random.randint(50, 500),
                random.randint(10, 100), f'https://demo.com/{i}', 'demo'
            )
            if question_data:
                all_questions.append(question_data)
        
        return self._create_dataframe(all_questions)
    
    def _is_question(self, text: str) -> bool:
        """Check if text is a question"""
        if not text or len(text) < CONFIG['min_question_length']:
            return False
        
        text = text.strip()
        
        # Must end with question mark or start with question word
        if text.endswith('?'):
            return True
        
        question_starters = ['what', 'how', 'why', 'when', 'where', 'which', 'who', 'do', 'are', 'have', 'can', 'will', 'would']
        if any(text.lower().startswith(starter) for starter in question_starters):
            return True
        
        return False
    
    def _extract_questions_from_text(self, text: str) -> List[str]:
        """Extract questions from a block of text"""
        sentences = re.split(r'[.!?]+', text)
        questions = []
        
        for sentence in sentences:
            sentence = sentence.strip()
            if self._is_question(sentence + '?'):  # Add ? for testing
                questions.append(sentence + ('?' if not sentence.endswith('?') else ''))
        
        return questions
    
    def _process_question(self, question: str, subreddit: str, score: int, 
                         comments: int, url: str, source_type: str) -> Optional[Dict]:
        """Process and score a question"""
        question = question.strip()
        
        # Basic filtering
        if len(question) < CONFIG['min_question_length'] or len(question) > CONFIG['max_question_length']:
            return None
        
        # Check for dating relevance
        if not self._is_dating_relevant(question):
            return None
        
        # Calculate hybrid score
        scores = self.scoring_model.calculate_hybrid_score(question)
        
        # Filter by minimum score
        if scores['hybrid_score'] < CONFIG['min_hybrid_score']:
            return None
        
        # Generate unique ID
        question_id = hashlib.md5(question.encode()).hexdigest()[:8]
        
        # Determine theme
        theme = self._determine_theme(question)
        
        return {
            'question_id': question_id,
            'question': question,
            'theme': theme,
            'reddit_topic': subreddit,
            'score': scores['hybrid_score'],
            'timestamp': datetime.now().isoformat(),
            'similarity_to_core': scores['similarity_to_core'],
            'framework_match': scores['framework_match'],
            'depth_potential': scores['depth_potential'],
            'engagement_potential': scores['engagement_potential'],
            'personal_connection': scores['personal_connection'],
            'reddit_score': score,
            'reddit_comments': comments,
            'reddit_url': url,
            'source_type': source_type,
            'emojis_present': bool(re.search(r'[😀-🿿]', question)),
            'is_fun_content': 'fun' in question.lower() or 'game' in question.lower()
        }
    
    def _is_dating_relevant(self, question: str) -> bool:
        """Check if question is relevant to dating"""
        question_lower = question.lower()
        
        # Direct keyword match
        if any(keyword in question_lower for keyword in DATING_KEYWORDS):
            return True
        
        # Relationship-related patterns
        relationship_patterns = [
            r'\b(partner|significant other|so)\b',
            r'\b(first date|dating|relationship)\b',
            r'\b(love|romance|romantic)\b',
            r'\b(attraction|attracted)\b'
        ]
        
        if any(re.search(pattern, question_lower) for pattern in relationship_patterns):
            return True
        
        # Personal connection questions (often good for dating)
        personal_patterns = [
            r'what.*you.*like',
            r'how.*you.*feel',
            r'what.*your.*favorite',
            r'tell me about',
            r'what.*makes you'
        ]
        
        if any(re.search(pattern, question_lower) for pattern in personal_patterns):
            return True
        
        return False
    
    def _determine_theme(self, question: str) -> str:
        """Determine the theme of a question"""
        question_lower = question.lower()
        
        if any(word in question_lower for word in ['love', 'romance', 'romantic', 'heart']):
            return 'Romance & Love'
        elif any(word in question_lower for word in ['date', 'dating', 'first date']):
            return 'Dating & Courtship'
        elif any(word in question_lower for word in ['relationship', 'partner', 'together']):
            return 'Relationships'
        elif any(word in question_lower for word in ['personal', 'yourself', 'about you']):
            return 'Personal & Identity'
        elif any(word in question_lower for word in ['future', 'goal', 'dream', 'plan']):
            return 'Future & Goals'
        elif any(word in question_lower for word in ['fun', 'enjoy', 'hobby', 'interest']):
            return 'Interests & Hobbies'
        elif any(word in question_lower for word in ['family', 'friend', 'social']):
            return 'Social & Family'
        else:
            return 'General'
    
    def _create_dataframe(self, questions: List[Dict]) -> pd.DataFrame:
        """Create and clean the final dataframe"""
        if not questions:
            print("❌ No questions found!")
            return pd.DataFrame()
        
        df = pd.DataFrame(questions)
        
        # Remove duplicates
        df = df.drop_duplicates(subset=['question'], keep='first')
        
        # Sort by score
        df = df.sort_values('score', ascending=False)
        
        # Limit to max questions
        if len(df) > CONFIG['max_questions_total']:
            df = df.head(CONFIG['max_questions_total'])
        
        if CONFIG['verbose']:
            print(f"✅ Final dataset: {len(df)} high-quality questions")
            print(f"📊 Average score: {df['score'].mean():.1f}")
            print(f"🎯 Themes: {df['theme'].value_counts().to_dict()}")
        
        return df
    
    def save_results(self, df: pd.DataFrame) -> str:
        """Save results to Excel file"""
        if df.empty:
            print("❌ No data to save!")
            return ""
        
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        filename = f"hybrid_reddit_questions_{timestamp}.xlsx"
        filepath = Path(CONFIG['output_dir']) / filename
        
        try:
            # Save to Excel
            df.to_excel(filepath, index=False)
            
            # Also save as CSV for compatibility
            csv_filepath = filepath.with_suffix('.csv')
            df.to_csv(csv_filepath, index=False)
            
            print(f"✅ Results saved to:")
            print(f"   📊 Excel: {filepath}")
            print(f"   📄 CSV: {csv_filepath}")
            
            return str(filepath)
        
        except Exception as e:
            print(f"❌ Error saving results: {e}")
            return ""

print("✅ RedditExtractor class defined successfully!")

✅ RedditExtractor class defined successfully!


## Step 6: Run the Extraction

Execute this cell to start the question extraction process.

In [13]:
# Initialize and run the extractor
print("🚀 Ultimate Hybrid Reddit Dating Questions Extractor")
print("=" * 60)

# Initialize extractor
extractor = RedditExtractor(REDDIT_CONFIG)

# Extract questions
print("\n🔍 Starting question extraction...")
df = extractor.extract_questions()

if not df.empty:
    # Save results
    print("\n💾 Saving results...")
    filepath = extractor.save_results(df)
    
    # Display summary
    print(f"\n📈 Extraction Summary:")
    print(f"   Total questions: {len(df)}")
    print(f"   Average score: {df['score'].mean():.1f}")
    print(f"   Top themes: {', '.join(df['theme'].value_counts().head(3).index.tolist())}")
    print(f"   Score range: {df['score'].min():.1f} - {df['score'].max():.1f}")
    
    # Show top 5 questions
    print(f"\n🏆 Top 5 Questions:")
    for i, row in df.head(5).iterrows():
        print(f"   {i+1}. {row['question']} (Score: {row['score']:.1f})")
    
    print(f"\n✅ Extraction completed successfully!")
    print(f"📁 Files saved to: {CONFIG['output_dir']}")
else:
    print("❌ No questions extracted!")

🚀 Ultimate Hybrid Reddit Dating Questions Extractor
🤖 Hybrid model initialized with 20 core questions

🔍 Starting question extraction...
🔄 Using demo mode - generating sample questions
✅ Final dataset: 4 high-quality questions
📊 Average score: 72.5
🎯 Themes: {'Relationships': 2, 'Romance & Love': 1, 'Dating & Courtship': 1}

💾 Saving results...
✅ Results saved to:
   📊 Excel: outputs/hybrid_reddit_questions_20250617_173457.xlsx
   📄 CSV: outputs/hybrid_reddit_questions_20250617_173457.csv

📈 Extraction Summary:
   Total questions: 4
   Average score: 72.5
   Top themes: Relationships, Romance & Love, Dating & Courtship
   Score range: 65.6 - 79.2

🏆 Top 5 Questions:
   1. What's your love language and how did you discover it? (Score: 79.2)
   3. What's the best relationship advice you've ever received? (Score: 77.7)
   2. What's your ideal way to spend a weekend with a partner? (Score: 67.5)
   4. What's your biggest fear when it comes to dating? (Score: 65.6)

✅ Extraction completed s

## Step 7: View Results (Optional)

Display the extracted questions in the notebook for review.

In [14]:
# Display results in notebook
if 'df' in locals() and not df.empty:
    print("📊 Extracted Questions Summary:")
    print(f"Total: {len(df)} questions")
    print(f"Average Score: {df['score'].mean():.1f}")
    print("\nTheme Distribution:")
    print(df['theme'].value_counts())
    
    print("\n📋 Sample Questions:")
    # Display first 10 questions
    display_df = df[['question', 'theme', 'score', 'reddit_topic']].head(10)
    display(display_df)
else:
    print("❌ No data to display. Please run the extraction first.")

📊 Extracted Questions Summary:
Total: 4 questions
Average Score: 72.5

Theme Distribution:
Relationships         2
Romance & Love        1
Dating & Courtship    1
Name: theme, dtype: int64

📋 Sample Questions:


,question,theme,score,reddit_topic
0,What's your love language and how did you disc...,Romance & Love,79.25,demo
2,What's the best relationship advice you've eve...,Relationships,77.69,demo
1,What's your ideal way to spend a weekend with ...,Relationships,67.49,demo
3,What's your biggest fear when it comes to dating?,Dating & Courtship,65.57,demo


## 🎉 Extraction Complete!

### What You Get:
- **High-quality dating questions** with scores ≥65
- **All required columns**: question_id, question, theme, reddit_topic, score, timestamp
- **Detailed scoring breakdown** for each question
- **Excel and CSV files** for easy use
- **Smart filtering** for dating relevance

### Output Files:
- Excel: `outputs/hybrid_reddit_questions_YYYYMMDD_HHMMSS.xlsx`
- CSV: `outputs/hybrid_reddit_questions_YYYYMMDD_HHMMSS.csv`

### Next Steps:
1. Review the extracted questions
2. Adjust CONFIG settings if needed
3. Re-run extraction with different parameters
4. Use the questions in your dating app!

### Configuration Tips:
- Lower `min_hybrid_score` to get more questions
- Increase `posts_per_subreddit` for more comprehensive search
- Add Reddit API credentials for real Reddit data
- Modify `DATING_KEYWORDS` for different focus areas